In [6]:
import os
import random
import time
import h5py
import numpy as np
import pandas as pd
import scipy.io
import librosa
import kagglehub
import h5py
import shutil
import IPython.display as ipd

from IPython.display import display, Audio
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score 
from sktime.classification.kernel_based import RocketClassifier
from sktime.transformations.panel.rocket import Rocket


In [2]:
# ==================================================
# 2. CARGA Y CONCATENACIÓN DE BABBLE NOISE (LOCAL)
# ==================================================
babble_exact_path = "datasets/noise/0dB/valid/real" 

audio_buffers_list = []
max_archivos_a_combinar = 40 
sr_original = 16000
sr_objetivo = 22050

archivos_mat = 0

for f in os.listdir(babble_exact_path):
    if len(audio_buffers_list) >= max_archivos_a_combinar:
        break

    file_path = os.path.join(babble_exact_path, f)
    
    if file_path.endswith('.mat'):
            mat_contents = scipy.io.loadmat(file_path) #esta funcion convierte el .mat en un diccionario 
            keys = [k for k in mat_contents.keys() if not k.startswith('__')] # cuando SciPy lee un .mat, automáticamente le inyecta variables internas de configuración que siempre empiezan con doble guion bajo (ej: __header. 
            #armo una lista solo con los nombres de las variables "reales" que contienen el audio
            if keys:
                audio_raw = mat_contents[keys[0]].flatten().astype(np.float32) 
                #mat_contents[keys[0]] extrae los datos almacenados en la primera clave válida
                # .flatten(): "aplasta" esa matriz para convertirla en un vector 1D
                # .astype(np.float32): conversión a formato decimal de 32 bits (float32).
                if audio_raw.size > 0:
                    audio_buffers_list.append(audio_raw)
                    archivos_mat += 1

# Procesamiento final
print("-" * 50)
print("Estadísticas de carga directa:")
print(f" - Archivos .mat procesados con éxito: {archivos_mat}")

babble_completo_16k = np.concatenate(audio_buffers_list)
babble_audio_full = librosa.resample(babble_completo_16k, orig_sr=sr_original, target_sr=sr_objetivo) #la paso a la sample rate que necesito

--------------------------------------------------
Estadísticas de carga directa:
 - Archivos .mat procesados con éxito: 40


In [3]:
# ==================================================
# 3. FUNCIONES DE DATA AUGMENTATION
# ==================================================
def add_white_noise(audio, noise_level=0.005):
    noise = np.random.randn(len(audio)) #vector de numeros aleatorios con la misma longitud que mi vector original
    return audio + noise_level * noise

def add_pink_noise(audio, noise_level=0.01):
    #el ruido rosa se caracteriza por tener una densidad espectral inversa a la frecuencia (es decir, 1/f)
    white = np.random.randn(len(audio)) #genero ruido blanco
    fft_white = np.fft.rfft(white) #transformada de fourier - paso el ruido blanco a dominio de la frecuencia. Ya no es la potencia, es la amplitud la que veo representada
    frequencies = np.maximum(np.fft.rfftfreq(len(audio)), 1e-10) #crea un vector de frecuencias. SI hay un valor menor a 1e-10, lo reemplaza por este número.
    #Elijo 1e-10 porque es un numero cercano a cero, sin ser cero.
    f_filter = 1.0 / np.sqrt(frequencies) #En la línea anterior, me aseguré de que no queden divisiones por cero. V^2 = P -> 1/sqrt(f) = P (ver notas)
    f_filter /= np.max(f_filter) #Busca el numero mas grande que haya quedado dentro del array y divide a todos los valores por este numero
    fft_pink = fft_white * f_filter #Aplica el filtro al ruido blanco en el dominio de la frecuencia.
    pink = np.fft.irfft(fft_pink, n=len(audio)) #Conversion al dominio del tiempo
    pink = pink / np.max(np.abs(pink)) #normalizacion
    return audio + noise_level * pink

def add_babble_noise(audio, babble_audio, noise_level=0.03):
    # np.tile -> construye un nuevo array repitiendo el primer argumento (en este caso, babble_audio) la cantidad de veces que se pida. 
    # Esto se aplica solo si el audio es menor al audio original  
    #np.ceil -> devuelve el menor escalar i tal que i>=x
    if len(babble_audio) < len(audio):
        babble_audio = np.tile(babble_audio, int(np.ceil(len(audio) / len(babble_audio))))

    # toma un punto de inicio random
    start_idx = random.randint(0, len(babble_audio) - len(audio))
    babble_chunk = babble_audio[start_idx : start_idx + len(audio)] #aplico babble noise desde mi punto random hasta el final del audio
    babble_chunk = babble_chunk / (np.max(np.abs(babble_chunk)) + 1e-10) #normalizo el murmullo para que no haya picos de volumen descontrolados

    return audio + noise_level * babble_chunk 


In [18]:
# ==========================================
# 4. DESCARGA Y CURACIÓN DEL DATASET LOCAL
# ==========================================
dataset_root_path = kagglehub.dataset_download("mmoreaux/environmental-sound-classification-50")

metadata_path = None
for root, dirs, files in os.walk(dataset_root_path):
    if 'esc50.csv' in files:
        metadata_path = os.path.join(root, 'esc50.csv')
        break

if metadata_path:
    df = pd.read_csv(metadata_path)
    print("Archivo de metadatos cargado correctamente.")

mis_clases = ['clock_alarm', 'door_wood_knock', 'cat', 'crying_baby', 'dog', 'glass_breaking']
df_filtrado = df[df['category'].isin(mis_clases)].copy()

kaggle_audio_dir = None
for root, dirs, files in os.walk(dataset_root_path): #Busca donde estan los archivos
    if any(f.endswith('.wav') for f in files):
        kaggle_audio_dir = root
        break

target_dir = "datasets/audios_originales"
archivos_copiados = 0

for index, row in df_filtrado.iterrows():
    src_path = os.path.join(kaggle_audio_dir, row['filename'])
    dst_path = os.path.join(target_dir, row['filename'])
    
    # Solo lo copio si el archivo original existe y no esta en el directorio audios_originales
    if os.path.exists(src_path) and not os.path.exists(dst_path):
        shutil.copy2(src_path, dst_path)
        archivos_copiados += 1

print(f"Se copiaron {archivos_copiados} archivos nuevos.")


Archivo de metadatos cargado correctamente.
Se copiaron 0 archivos nuevos.


In [ ]:

# ==========================================
# 5. TRANSFORMACIÓN Y DATA AUGMENTATION
# ==========================================
correct_audio_dir = target_dir 

X_list = []
y_list = []
groups = []

for index, row in df_filtrado.iterrows():
    file_path = os.path.join(correct_audio_dir, row['filename'])
    categoria = row['category']

    try:
        y_audio, sr = librosa.load(file_path, sr=22050)

        audios_a_procesar = [
            y_audio,                                     
            add_white_noise(y_audio),                    
            add_pink_noise(y_audio),                     
            add_babble_noise(y_audio, babble_audio_full) 
        ]

        for audio_version in audios_a_procesar:
            X_list.append(audio_version)
            y_list.append(categoria)
            groups.append(row['filename'])
        
    except Exception as e:
        print(f"⚠️ Error procesando {row['filename']}: {e}")

X = np.array(X_list)
y = np.array(y_list)
groups = np.array(groups)

print("-" * 50)
print("✅ Extracción y aumentación finalizada.")
print(f"Forma final de la matriz X: {X.shape}")
print(f"Cantidad de etiquetas y: {y.shape}")


--------------------------------------------------
✅ Extracción y aumentación finalizada.
Forma final de la matriz X: (960, 110250)
Cantidad de etiquetas y: (960,)


In [16]:
# ==========================================
# 4. TRANSFORMACIÓN Y DATA AUGMENTATION
# ==========================================
correct_audio_dir = None
dataset_root_path = kagglehub.dataset_download("mmoreaux/environmental-sound-classification-50")
metadata_path = os.path.join(dataset_root_path, 'esc50.csv')

if os.path.exists(metadata_path):
    df = pd.read_csv(metadata_path)
    print("Archivo de metadatos cargado correctamente.")
else:
    raise FileNotFoundError(f"No se encontró el archivo esc50.csv en la ruta: {metadata_path}")

# Mis clases
mis_clases = ['alarm', 'door_bell', 'cat', 'crying_baby', 'dog', 'shouting']
df_filtrado = df[df['category'].isin(mis_clases)].copy()


for root, dirs, files in os.walk(dataset_root_path):
    if any(f.endswith('.wav') for f in files):
        correct_audio_dir = root
        break

X_list = []
y_list = []

print(f"DataFrame listo: {len(df_filtrado)} audios base listos para ser multiplicados.")

for index, row in df_filtrado.iterrows():
    file_path = os.path.join(correct_audio_dir, row['filename'])
    categoria = row['category']

    try:
        y_audio, sr = librosa.load(file_path, sr=22050)
        # print(y_audio.shape)

        audios_a_procesar = [
            y_audio,                                     
            add_white_noise(y_audio),                    
            add_pink_noise(y_audio),                     
            add_babble_noise(y_audio, babble_audio_full) 
        ]

        for audio_version in audios_a_procesar:
            X_list.append(audio_version)
            y_list.append(categoria) 
        
    except:
        print("blabla")
X = np.array(X_list)
y = np.array(y_list)
print(X.shape)


Archivo de metadatos cargado correctamente.
DataFrame listo: 120 audios base listos para ser multiplicados.
(480, 110250)


In [17]:
y

array(['dog', 'dog', 'dog', 'dog', 'dog', 'dog', 'dog', 'dog',
       'crying_baby', 'crying_baby', 'crying_baby', 'crying_baby',
       'crying_baby', 'crying_baby', 'crying_baby', 'crying_baby',
       'crying_baby', 'crying_baby', 'crying_baby', 'crying_baby',
       'crying_baby', 'crying_baby', 'crying_baby', 'crying_baby',
       'crying_baby', 'crying_baby', 'crying_baby', 'crying_baby',
       'crying_baby', 'crying_baby', 'crying_baby', 'crying_baby', 'dog',
       'dog', 'dog', 'dog', 'dog', 'dog', 'dog', 'dog', 'dog', 'dog',
       'dog', 'dog', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat',
       'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat',
       'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat', 'cat',
       'cat', 'cat', 'cat', 'dog', 'dog', 'dog', 'dog', 'crying_baby',
       'crying_baby', 'crying_baby', 'crying_baby', 'crying_baby',
       'crying_baby', 'crying_baby', 'crying_baby', 'cat', 'cat', 'cat',
       'cat', 'dog', 'dog', 'dog', 'd

In [19]:
# ==========================================
# 5. ENTRENAMIENTO CON DETACH-ROCKET
# ==========================================
from sklearn.ensemble import RandomForestClassifier
le = LabelEncoder()
y_encoded = le.fit_transform(y)


X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
print("-" * 50)
print("RESUMEN DE AUDIOS UTILIZADOS:")
print(f"Total de muestras (con Data Augmentation): {len(X)}")
print(f" - Muestras para Entrenamiento: {len(X_train)}")
print(f" - Muestras para Prueba (Test): {len(X_test)}")
print("-" * 50)

inicio = time.time()

# Instanciar y ajustar Detach-ROCKET
detach_model = RocketClassifier(num_kernels=500)
detach_model.fit(X_train, y_train)

--------------------------------------------------
RESUMEN DE AUDIOS UTILIZADOS:
Total de muestras (con Data Augmentation): 480
 - Muestras para Entrenamiento: 384
 - Muestras para Prueba (Test): 96
--------------------------------------------------


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


RocketClassifier(num_kernels=500)

In [20]:

# Hacer predicciones sobre el conjunto de prueba
y_pred = detach_model.predict(X_test)

# Calcular la métrica con sklearn
accuracy = accuracy_score(y_test, y_pred)


In [31]:
accuracy

1.0

In [ ]:



# Hacer predicciones sobre el conjunto de prueba
y_pred = detach_model.predict(X_test)

# Calcular la métrica con sklearn
accuracy = accuracy_score(y_test, y_pred)
fin = time.time()

print(f"Precisión (Test Accuracy): {accuracy * 100:.2f}%")
print(f"Tiempo total de ejecución: {fin - inicio:.2f} segundos")

# DetachRocket permite visualizar qué proporción de variables (features) retuvo:
if hasattr(detach_model, 'feature_proportion_'):
    print(f"Porcentaje de features retenidas tras el podado: {detach_model.feature_proportion_ * 100:.2f}%")

# ==========================================
# 6. AUDICIÓN DE VARIANTES
# ==========================================
if 'df_filtrado' in locals() and 'correct_audio_dir' in locals() and correct_audio_dir is not None:
    random_row = df_filtrado.sample(n=1).iloc[0]
    random_file_path = os.path.join(correct_audio_dir, random_row['filename'])
    random_label = random_row['category']

    print("=" * 50)
    print(f"AUDICIÓN DE VARIANTES: {random_label.upper()}")
    print(f"Archivo base: {random_row['filename']}")
    print("=" * 50)

    try:
        y_audio_test, sr_audio = librosa.load(random_file_path, sr=22050)
        
        print("\n1. Audio Original:")
        display(ipd.Audio(y_audio_test, rate=sr_audio))
        
        print("\n2. Variante: Ruido Blanco:")
        y_white = add_white_noise(y_audio_test)
        display(ipd.Audio(y_white, rate=sr_audio))
            
        print("\n3. Variante: Ruido Rosa:")
        y_pink = add_pink_noise(y_audio_test)
        display(ipd.Audio(y_pink, rate=sr_audio))
            
        if babble_audio_full is not None:
            print("\n4. Variante: Murmullo de fondo (Babble Noise):")
            y_babble = add_babble_noise(y_audio_test, babble_audio_full)
            display(ipd.Audio(y_babble, rate=sr_audio))
        else:
            print("\n[Aviso: No se generó Babble Noise válido]")
        
    except Exception as e:
        print(f"Error interno al intentar procesar los audios: {e}")

--------------------------------------------------
RESUMEN DE AUDIOS UTILIZADOS:
Total de muestras (con Data Augmentation): 480
 - Muestras para Entrenamiento: 384
 - Muestras para Prueba (Test): 96
--------------------------------------------------


ValueError: Found input variables with inconsistent numbers of samples: [1, 384]